# 🔐 Mobile Biometrics - Interactive Training Notebook

Bu notebook, projedeki ML pipeline'ını adım adım incelemenizi sağlar.

**Jupyter'ın Avantajları:**
- Hücreleri tek tek çalıştırabilirsiniz (Shift+Enter)
- Ara sonuçları görebilirsiniz
- Grafikleri inline görürsünüz
- Markdown ile açıklama ekleyebilirsiniz

## 1. Kütüphaneleri Yükle

İlk hücreyi çalıştırarak başlayın. Eğer hata alırsanız, hücreyi düzeltip tekrar çalıştırabilirsiniz - tüm scripti baştan çalıştırmanıza gerek yok!

In [ ]:
import sys
sys.path.append('../src')  # src modüllerini kullanabilmek için

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Jupyter'da grafikleri inline göster
%matplotlib inline

# Grafik ayarları
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✅ Kütüphaneler yüklendi!")

## 2. Veriyi Yükle ve İncele

Aşağıda session features'ı yükleyip ilk 5 satırı görüyoruz. Script'te bunu göremezdiniz - sadece print edilir ve geçerdi.

In [ ]:
# Session features yükle
df = pd.read_csv('../output/features/session_features.csv')

print(f"📊 Dataset boyutu: {df.shape}")
print(f"👥 Kullanıcı sayısı: {df['user'].nunique()}")
print(f"📁 Session sayısı: {len(df)}")

# İlk 5 satır - Jupyter'da güzel tablo olarak görünür!
df.head()

☝️ **Fark ettiniz mi?** DataFrame otomatik olarak güzel bir tablo olarak görüntülendi. Python script'te bunu göremezdiniz!

In [ ]:
# Veri özeti - describe() ile istatistikler
df.describe()

## 3. Görselleştirme (Inline Grafikler)

Script'te grafikler ayrı pencerede açılır. Jupyter'da kodun hemen altında görünür!

In [ ]:
# Dwell time dağılımı
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Dwell Time
axes[0].hist(df['dwell_time_mean'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Dwell Time Distribution')
axes[0].set_xlabel('Mean Dwell Time (ms)')

# Flight Time
axes[1].hist(df['flight_time_mean'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Flight Time Distribution')
axes[1].set_xlabel('Mean Flight Time (ms)')

# Inter-key Latency
axes[2].hist(df['inter_key_latency_mean'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='green')
axes[2].set_title('Inter-key Latency Distribution')
axes[2].set_xlabel('Mean Latency (ms)')

plt.tight_layout()
plt.show()

## 4. Deneme ve Hata Yapma

Aşağıdaki hücreyi farklı parametrelerle tekrar tekrar çalıştırabilirsiniz. Script'te her seferinde baştan çalıştırmanız gerekirdi!

In [ ]:
# 🔧 Bu değeri değiştirip tekrar çalıştırın (Shift+Enter)!
SELECTED_USER = 'AA'

user_data = df[df['user'] == SELECTED_USER]
print(f"📌 {SELECTED_USER} kullanıcısı için {len(user_data)} session var")
print(f"   Ortalama dwell time: {user_data['dwell_time_mean'].mean():.2f} ms")
print(f"   Ortalama flight time: {user_data['flight_time_mean'].mean():.2f} ms")

## 5. Model Eğitimi (İnteraktif)

Modül import edip hızlıca test edebiliriz.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Features ve labels
feature_cols = [c for c in df.columns if c not in ['user', 'session']]
X = df[feature_cols].fillna(0)
y = df['user']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# 🔧 n_estimators değerini değiştirip tekrar çalıştırın!
N_ESTIMATORS = 100

model = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"🎯 n_estimators={N_ESTIMATORS} ile Accuracy: {accuracy*100:.2f}%")

## 6. Feature Importance (Görsel)

Hangi özellikler en önemli? Jupyter'da hemen görselleştirebiliriz.

In [ ]:
# Feature importance
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 15
plt.figure(figsize=(10, 6))
plt.barh(importances['feature'][:15][::-1], importances['importance'][:15][::-1])
plt.xlabel('Importance')
plt.title('Top 15 Most Important Features')
plt.tight_layout()
plt.show()

# Tablo olarak da göster
importances.head(10)

## 📝 Özet: Jupyter vs Python Script

| Jupyter Notebook | Python Script |
|------------------|---------------|
| ✅ Hücre hücre çalıştırma | ❌ Tümü birden çalışır |
| ✅ Grafikler inline | ❌ Ayrı pencere |
| ✅ DataFrame güzel görünür | ❌ Print edilir |
| ✅ Deneme/hata kolay | ❌ Baştan çalıştır |
| ❌ Git diff zor | ✅ Git diff kolay |
| ❌ Production için uygun değil | ✅ Production ready |

**Sonuç:** İkisi de farklı amaçlara hizmet eder!
- **Keşif & Eğitim** → Jupyter
- **Production & Pipeline** → Python Script

In [ ]:
print("🎉 Notebook tamamlandı!")